In [1]:
import re

import numpy as np
import pandas as pd

In [2]:
SEED_FILE = "Synthetic Dataset.csv"
DATASETS = {
    "Rule-based (mine)": "synthetic_students.csv",
    "Seed-based (yours)": "my_synthetic_students_data.csv",
}

In [3]:
REPORT_FILE = "privacy_report.txt"

NEAR_COPY_DISTANCE = 0.10   # closer than this (standardised units) = near-copy
MAX_NEAR_COPY_PCT = 0.0     # warn if ANY row is a near-copy of a seed row
K_MIN = 3                   # a row should look like at least 3 students
MAX_RISKY_PCT = 10.0        # warn if more than 10% of rows have k < K_MIN
ID_PATTERN = r"^S\d{4,}$"   # valid fake IDs look like S0001

NUMERIC = ["Age", "Study_hrs", "Attendance%", "Total_screen_time",
           "Sleep_hrs", "exam_score"]
STANDARD = ["ID", "Age", "Study_hrs", "Attendance%", "Total_screen_time",
            "Sleep_hrs", "Internet_quality", "exam_score", "Result"]

EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE = re.compile(r"\+?\d[\d\s().-]{8,}\d")

lines = []   # everything we print is also saved here

In [4]:
def log(text=""):
    print(text)
    lines.append(text)

In [5]:
def load(path):
    df = pd.read_csv(path)
    lookup = {c.lower(): c for c in STANDARD}
    df.columns = [lookup.get(c.strip().replace(" ", "_").lower(), c.strip())
                  for c in df.columns]
    q = df["Internet_quality"]
    if pd.api.types.is_numeric_dtype(q):
        df["Internet_quality"] = pd.cut(
            q, bins=[0, 2, 3, 5], labels=["Poor", "Average", "Good"]
        ).astype(str)
    else:
        df["Internet_quality"] = q.astype(str).str.strip().str.title()
    return df


In [6]:
def check_duplicates(df, issues):
    dup_ids = int(df["ID"].duplicated().sum())
    dup_rows = int(df.drop(columns=["ID"]).duplicated().sum())
    log(f"  Duplicate IDs: {dup_ids}")
    log(f"  Duplicate rows (ignoring ID): {dup_rows}")
    if dup_ids:
        issues.append((f"{dup_ids} duplicate IDs",
                       "Regenerate IDs so each one is unique."))
    if dup_rows:
        issues.append((f"{dup_rows} duplicate rows",
                       "Drop duplicates with df.drop_duplicates() or "
                       "regenerate with a different seed."))


In [7]:
def check_seed_copies(df, seed, issues):
    # Exact copies: every numeric value and the internet label match
    cols = NUMERIC + ["Internet_quality"]
    left = df[cols].astype({c: float for c in NUMERIC})
    right = seed[cols].astype({c: float for c in NUMERIC}).drop_duplicates()
    merged = left.merge(right, on=cols, how="inner")
    exact = len(merged)
    log(f"  Exact copies of seed rows: {exact}")
    if exact:
        issues.append((f"{exact} rows are exact copies of seed rows",
                       "Remove those rows and regenerate. Consider adding a "
                       "little random noise to numeric columns."))

    # Near-copies: nearest-neighbour distance in standardised units
    both = pd.concat([df[NUMERIC], seed[NUMERIC]])
    std = both.std().replace(0, 1)
    a = ((df[NUMERIC] - both.mean()) / std).to_numpy()
    b = ((seed[NUMERIC] - both.mean()) / std).to_numpy()
    dist = np.sqrt(((a[:, None, :] - b[None, :, :]) ** 2).sum(axis=2))
    nearest = dist.min(axis=1)

    # Baseline: how close are seed rows to each other?
    sd = np.sqrt(((b[:, None, :] - b[None, :, :]) ** 2).sum(axis=2))
    np.fill_diagonal(sd, np.inf)
    seed_nearest = sd.min(axis=1)

    near = int((nearest < NEAR_COPY_DISTANCE).sum())
    pct = 100 * near / len(df)
    log(f"  Nearest distance to a seed row: min {nearest.min():.2f}, "
        f"median {np.median(nearest):.2f}")
    log(f"  (For comparison, seed rows are a median of "
        f"{np.median(seed_nearest):.2f} from each other)")
    log(f"  Near-copies (distance < {NEAR_COPY_DISTANCE}): "
        f"{near} rows ({pct:.1f}%)")
    if pct > MAX_NEAR_COPY_PCT:
        issues.append((f"{near} rows ({pct:.1f}%) are near-copies of seed rows",
                       "Increase the noise in generation, use more seed rows, "
                       "or remove the flagged rows."))

In [8]:
def check_uniqueness(df, issues):
    d = df.copy()
    d["age_band"] = pd.cut(d["Age"], [0, 19, 22, 25, 30, 120])
    d["study_band"] = pd.cut(d["Study_hrs"], list(range(0, 45, 5)),
                             include_lowest=True)
    keys = ["age_band", "study_band", "Internet_quality"]
    k = d.groupby(keys, observed=True)["ID"].transform("count")
    risky = int((k < K_MIN).sum())
    pct = 100 * risky / len(d)
    unique = int((k == 1).sum())
    log(f"  Smallest group size (k): {int(k.min())}")
    log(f"  Rows that are unique on age band + study band + internet: {unique}")
    log(f"  Rows in groups smaller than {K_MIN}: {risky} ({pct:.1f}%)")
    if pct > MAX_RISKY_PCT:
        issues.append((f"{pct:.1f}% of rows are in groups smaller than {K_MIN}",
                       "Use wider bands (e.g. 10-year age groups) when "
                       "sharing, or remove/merge rare combinations."))

In [9]:
def check_identifiers(df, issues):
    bad_ids = int((~df["ID"].astype(str).str.match(ID_PATTERN)).sum())
    log(f"  IDs not matching {ID_PATTERN}: {bad_ids}")
    if bad_ids:
        issues.append((f"{bad_ids} IDs do not look like fake IDs",
                       "Replace them with generated IDs like S0001."))

    found = 0
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    for col in text_cols:
        text = df[col].astype(str)
        found += int(text.str.contains(EMAIL).sum())
        if col != "ID":
            found += int(text.str.contains(PHONE).sum())
    log(f"  Emails/phone numbers found in text columns: {found}")
    if found:
        issues.append((f"{found} possible emails or phone numbers",
                       "Delete or mask those values before sharing."))

In [10]:
def main():
    seed = load(SEED_FILE)
    all_issues = {}

    for name, path in DATASETS.items():
        df = load(path)
        issues = []
        log("=" * 60)
        log(f"{name}: {path} ({len(df)} rows)")
        log("=" * 60)
        log("1. Duplicates")
        check_duplicates(df, issues)
        log("2. Similarity to seed rows")
        check_seed_copies(df, seed, issues)
        log("3. Uniqueness (re-identification risk)")
        check_uniqueness(df, issues)
        log("4. Direct identifiers")
        check_identifiers(df, issues)
        all_issues[name] = issues
        log()

    log("=" * 60)
    log("ISSUE LIST")
    log("=" * 60)
    for name, issues in all_issues.items():
        if not issues:
            log(f"{name}: no issues above the thresholds.")
            continue
        log(f"{name}: {len(issues)} issue(s)")
        for i, (problem, fix) in enumerate(issues, 1):
            log(f"  {i}. {problem}")
            log(f"     Suggested fix: {fix}")
    log()
    log("Note: these checks reduce risk but cannot prove privacy. They "
        "depend on the thresholds set at the top of this script.")

    with open(REPORT_FILE, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"\nReport saved to {REPORT_FILE}")


if __name__ == "__main__":
    main()

Rule-based (mine): synthetic_students.csv (100 rows)
1. Duplicates
  Duplicate IDs: 0
  Duplicate rows (ignoring ID): 0
2. Similarity to seed rows
  Exact copies of seed rows: 0
  Nearest distance to a seed row: min 0.62, median 1.73
  (For comparison, seed rows are a median of 1.80 from each other)
  Near-copies (distance < 0.1): 0 rows (0.0%)
3. Uniqueness (re-identification risk)
  Smallest group size (k): 1
  Rows that are unique on age band + study band + internet: 13
  Rows in groups smaller than 3: 37 (37.0%)
4. Direct identifiers
  IDs not matching ^S\d{4,}$: 0
  Emails/phone numbers found in text columns: 0

Seed-based (yours): my_synthetic_students_data.csv (500 rows)
1. Duplicates
  Duplicate IDs: 0
  Duplicate rows (ignoring ID): 0
2. Similarity to seed rows
  Exact copies of seed rows: 0
  Nearest distance to a seed row: min 0.56, median 1.41
  (For comparison, seed rows are a median of 1.51 from each other)
  Near-copies (distance < 0.1): 0 rows (0.0%)
3. Uniqueness (re-i